In [0]:
base_path = "/Volumes/workspace/retail_capstone/retail_sales_volume/"

display(dbutils.fs.ls(base_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/retail_capstone/retail_sales_volume/cleaned_sales.csv,cleaned_sales.csv,1088,1794995987000
dbfs:/Volumes/workspace/retail_capstone/retail_sales_volume/products.csv,products.csv,110,1794995987000


In [0]:
sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "cleaned_sales.csv")
)

display(sales_df)

sale_id,store_id,product_id,quantity,price,cost,discount,revenue,discount_amount,net_revenue,total_cost,profit,discount_percentage,profit_margin,product_name,category
1,101,201,10,500,300,20,5000,200,4800,3000,1800,4.0,37.5,Laptop,Electronics
2,101,202,5,1000,650,50,5000,250,4750,3250,1500,5.0,31.57894736842105,Headphones,Accessories
3,102,201,8,500,300,25,4000,200,3800,2400,1400,5.0,36.84210526315789,Laptop,Electronics
4,102,203,4,1500,1000,100,6000,400,5600,4000,1600,6.666666666666667,28.57142857142857,Keyboard,Accessories
5,103,202,12,1000,650,75,12000,900,11100,7800,3300,7.5,29.72972972972973,Headphones,Accessories
6,103,203,3,1500,1000,50,4500,150,4350,3000,1350,3.3333333333333335,31.03448275862069,Keyboard,Accessories
7,101,203,6,1500,1000,0,9000,0,9000,6000,3000,0.0,33.33333333333333,Keyboard,Accessories
8,102,202,7,1000,650,30,7000,210,6790,4550,2240,3.0,32.98969072164948,Headphones,Accessories
9,103,201,9,500,300,10,4500,90,4410,2700,1710,2.0,38.775510204081634,Laptop,Electronics
10,101,201,5,500,300,0,2500,0,2500,1500,1000,0.0,40.0,Laptop,Electronics


In [0]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "products.csv")
)

display(products_df)

product_id,product_name,category
201,Laptop,Electronics
202,Headphones,Accessories
203,Keyboard,Accessories


In [0]:
print("SALES DATA")
sales_df.printSchema()

print("PRODUCT DATA")
products_df.printSchema()

SALES DATA
root
 |-- sale_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- cost: integer (nullable = true)
 |-- discount: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- discount_amount: integer (nullable = true)
 |-- net_revenue: integer (nullable = true)
 |-- total_cost: integer (nullable = true)
 |-- profit: integer (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- profit_margin: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)

PRODUCT DATA
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)



In [0]:
print("Sales records:", sales_df.count())
print("Product records:", products_df.count())

print("Sales columns:", sales_df.columns)
print("Product columns:", products_df.columns)

Sales records: 10
Product records: 3
Sales columns: ['sale_id', 'store_id', 'product_id', 'quantity', 'price', 'cost', 'discount', 'revenue', 'discount_amount', 'net_revenue', 'total_cost', 'profit', 'discount_percentage', 'profit_margin', 'product_name', 'category']
Product columns: ['product_id', 'product_name', 'category']


In [0]:
sales_df = sales_df.withColumn(
    "product_id",
    sales_df["product_id"].cast("string")
)

products_df = products_df.withColumn(
    "product_id",
    products_df["product_id"].cast("string")
)

joined_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
)

display(joined_df)

product_id,sale_id,store_id,quantity,price,cost,discount,revenue,discount_amount,net_revenue,total_cost,profit,discount_percentage,profit_margin,product_name,category,product_name,category
201,1,101,10,500,300,20,5000,200,4800,3000,1800,4.0,37.5,Laptop,Electronics,Laptop,Electronics
202,2,101,5,1000,650,50,5000,250,4750,3250,1500,5.0,31.57894736842105,Headphones,Accessories,Headphones,Accessories
201,3,102,8,500,300,25,4000,200,3800,2400,1400,5.0,36.84210526315789,Laptop,Electronics,Laptop,Electronics
203,4,102,4,1500,1000,100,6000,400,5600,4000,1600,6.666666666666667,28.57142857142857,Keyboard,Accessories,Keyboard,Accessories
202,5,103,12,1000,650,75,12000,900,11100,7800,3300,7.5,29.72972972972973,Headphones,Accessories,Headphones,Accessories
203,6,103,3,1500,1000,50,4500,150,4350,3000,1350,3.3333333333333335,31.03448275862069,Keyboard,Accessories,Keyboard,Accessories
203,7,101,6,1500,1000,0,9000,0,9000,6000,3000,0.0,33.33333333333333,Keyboard,Accessories,Keyboard,Accessories
202,8,102,7,1000,650,30,7000,210,6790,4550,2240,3.0,32.98969072164948,Headphones,Accessories,Headphones,Accessories
201,9,103,9,500,300,10,4500,90,4410,2700,1710,2.0,38.775510204081634,Laptop,Electronics,Laptop,Electronics
201,10,101,5,500,300,0,2500,0,2500,1500,1000,0.0,40.0,Laptop,Electronics,Laptop,Electronics


In [0]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round as spark_round,
    when
)

sales_metrics = joined_df.withColumn(
    "total_revenue",
    col("quantity") * col("price")
).withColumn(
    "total_cost",
    col("quantity") * col("cost")
).withColumn(
    "profit",
    col("total_revenue") - col("total_cost")
)

display(sales_metrics)

product_id,sale_id,store_id,quantity,price,cost,discount,revenue,discount_amount,net_revenue,total_cost,profit,discount_percentage,profit_margin,product_name,category,product_name,category,total_revenue
201,1,101,10,500,300,20,5000,200,4800,3000,2000,4.0,37.5,Laptop,Electronics,Laptop,Electronics,5000
202,2,101,5,1000,650,50,5000,250,4750,3250,1750,5.0,31.57894736842105,Headphones,Accessories,Headphones,Accessories,5000
201,3,102,8,500,300,25,4000,200,3800,2400,1600,5.0,36.84210526315789,Laptop,Electronics,Laptop,Electronics,4000
203,4,102,4,1500,1000,100,6000,400,5600,4000,2000,6.666666666666667,28.57142857142857,Keyboard,Accessories,Keyboard,Accessories,6000
202,5,103,12,1000,650,75,12000,900,11100,7800,4200,7.5,29.72972972972973,Headphones,Accessories,Headphones,Accessories,12000
203,6,103,3,1500,1000,50,4500,150,4350,3000,1500,3.3333333333333335,31.03448275862069,Keyboard,Accessories,Keyboard,Accessories,4500
203,7,101,6,1500,1000,0,9000,0,9000,6000,3000,0.0,33.33333333333333,Keyboard,Accessories,Keyboard,Accessories,9000
202,8,102,7,1000,650,30,7000,210,6790,4550,2450,3.0,32.98969072164948,Headphones,Accessories,Headphones,Accessories,7000
201,9,103,9,500,300,10,4500,90,4410,2700,1800,2.0,38.775510204081634,Laptop,Electronics,Laptop,Electronics,4500
201,10,101,5,500,300,0,2500,0,2500,1500,1000,0.0,40.0,Laptop,Electronics,Laptop,Electronics,2500


In [0]:
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round, when

sales_metrics = joined_df.drop(products_df["category"])

sales_metrics = sales_metrics.withColumn(
    "total_revenue",
    col("quantity") * col("price")
).withColumn(
    "total_cost",
    col("quantity") * col("cost")
).withColumn(
    "profit",
    col("total_revenue") - col("total_cost")
)

category_metrics = (
    sales_metrics
    .groupBy("category")
    .agg(
        spark_sum("quantity").alias("total_quantity_sold"),
        spark_round(spark_sum("total_revenue"), 2).alias("total_revenue"),
        spark_round(spark_sum("total_cost"), 2).alias("total_cost"),
        spark_round(spark_sum("profit"), 2).alias("total_profit")
    )
    .withColumn(
        "profit_margin",
        when(
            col("total_revenue") != 0,
            spark_round(
                col("total_profit") / col("total_revenue") * 100, 2
            )
        ).otherwise(0)
    )
    .orderBy(col("total_profit").desc())
)

display(category_metrics)
   

category,total_quantity_sold,total_revenue,total_cost,total_profit,profit_margin
Accessories,37,43500,28600,14900,34.25
Electronics,32,16000,9600,6400,40.0


In [0]:
category_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_capstone.retail_category_metrics")

In [0]:
saved_metrics = spark.table(
    "workspace.retail_capstone.retail_category_metrics"
)

display(saved_metrics)

category,total_quantity_sold,total_revenue,total_cost,total_profit,profit_margin
Accessories,37,43500,28600,14900,34.25
Electronics,32,16000,9600,6400,40.0


In [0]:
sales_metrics = joined_df.drop(
    products_df["category"]
).drop(
    products_df["product_name"]
)

sales_metrics = sales_metrics.withColumn(
    "total_revenue",
    col("quantity") * col("price")
).withColumn(
    "total_cost",
    col("quantity") * col("cost")
).withColumn(
    "profit",
    col("total_revenue") - col("total_cost")
)

product_metrics = (
    sales_metrics
    .groupBy("product_id", "product_name")
    .agg(
        spark_sum("quantity").alias("total_quantity_sold"),
        spark_round(spark_sum("total_revenue"), 2).alias("total_revenue"),
        spark_round(spark_sum("profit"), 2).alias("total_profit")
    )
)

display(product_metrics)

product_id,product_name,total_quantity_sold,total_revenue,total_profit
202,Headphones,24,24000,8400
201,Laptop,32,16000,6400
203,Keyboard,13,19500,6500


In [0]:
product_metrics.createOrReplaceTempView("product_metrics")

print("Product metrics view created successfully!")

Product metrics view created successfully!


In [0]:
%sql
SELECT
    product_id,
    product_name,
    total_quantity_sold,
    total_revenue
FROM product_metrics
ORDER BY total_quantity_sold DESC
LIMIT 3;

product_id,product_name,total_quantity_sold,total_revenue
201,Laptop,32,16000
202,Headphones,24,24000
203,Keyboard,13,19500


In [0]:
product_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_capstone.retail_product_metrics")

print("Product metrics saved successfully!")

Product metrics saved successfully!
